
# Distribution Center 12-Month Financial Model

This Google Colab notebook uploads `DC_Forecast_Data.xlsx`, validates the workbook, builds Base/Upside/Downside forecasts, and produces:

- a CFO-ready standalone HTML report;
- an Excel output workbook with assumptions, calculations, scenarios, risks and validation checks;
- monthly charts for total cost, cost per case, required FTEs and capacity utilisation.

## Proposed modelling approach

1. **Validate inputs:** required sheets, exact columns, dates, duplicates, missing values, negative values and forecast horizon.
2. **Calibrate historical drivers:** labour productivity, labour cost rates, cases per transport load, transport cost per load, energy cost per pallet, maintenance cost per shipped case and storage pallet turns.
3. **Translate volume into operations:** forecast total processed cases, labour hours, FTEs, overtime, temporary labour, outbound loads and pallets stored.
4. **Forecast costs:**
   - **Fixed:** rent, baseline permanent labour and other operating costs.
   - **Variable:** temporary labour, overtime and transport.
   - **Semi-variable:** energy and maintenance, using simple historical cost-driver regressions with fixed and variable components.
5. **Run scenarios:** Base, Upside (+10% inbound and outbound volume) and Downside (-10%).
6. **Control and reconcile:** cost component tie-outs, capacity alerts, productivity reasonableness and monthly reconciliation checks.

## Important modelling assumptions and limitations

- The uploaded workbook uses cost columns ending in `_EUR`; the notebook validates those exact names.
- There is no opening inventory, target inventory days or closing inventory policy. Pallets stored are therefore estimated using historical pallet turns and the provided average cases per pallet.
- There is no temporary-labour-hours field. Temporary labour cost is estimated using permanent hourly cost plus a configurable agency premium.
- Standard hours per FTE and maximum overtime hours per FTE are not supplied, so editable defaults are used below.
- Annual inflation assumptions are compounded monthly from the final historical month.


In [ ]:

# STEP 1 — Install/import packages and upload the workbook
!pip -q install xlsxwriter plotly

from pathlib import Path
import io, math, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

try:
    from google.colab import files
    print('Upload DC_Forecast_Data.xlsx')
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('No file was uploaded.')
    INPUT_FILE = next(iter(uploaded.keys()))
except ImportError:
    # Local/Jupyter fallback: change this path only when not using Colab.
    INPUT_FILE = '/content/DC_Forecast_Data.xlsx'

OUTPUT_EXCEL = 'DC_12M_Financial_Model_Output.xlsx'
OUTPUT_HTML = 'DC_12M_CFO_Report.html'
print(f'Input file: {INPUT_FILE}')


In [ ]:

# STEP 2 — Assumptions, validation, calculations and outputs

# -----------------------------
# A. EDITABLE MODEL ASSUMPTIONS
# -----------------------------
MODEL_SETTINGS = {
    'standard_hours_per_fte_month': 160.0,
    'max_overtime_hours_per_fte_month': 20.0,
    'temporary_labour_premium_vs_permanent': 0.20,
    'baseline_fte_lookback_months': 3,
    'historical_driver_lookback_months': 12,
    'productivity_warning_low_factor': 0.75,
    'productivity_warning_high_factor': 1.25,
    'scenario_volume_change': {'Base': 0.00, 'Upside': 0.10, 'Downside': -0.10},
}

REQUIRED_COLUMNS = {
    'Historical_Operations': [
        'Month', 'Cases_Received', 'Cases_Shipped', 'Pallets_Stored',
        'Labour_Hours', 'FTE_Count', 'Overtime_Hours', 'Transport_Loads'
    ],
    'Historical_Costs': [
        'Month', 'Permanent_Labour_Cost_EUR', 'Temporary_Labour_Cost_EUR',
        'Overtime_Cost_EUR', 'Energy_Cost_EUR', 'Maintenance_Cost_EUR',
        'Rent_Cost_EUR', 'Transport_Cost_EUR', 'Other_Cost_EUR'
    ],
    'Forecast_Assumptions': [
        'Month', 'Forecast_Cases_Shipped', 'Forecast_Cases_Received',
        'Salary_Inflation_Percent', 'Energy_Inflation_Percent',
        'Transport_Inflation_Percent', 'Productivity_Cases_Per_Hour',
        'Maximum_Pallet_Capacity', 'Average_Cases_Per_Pallet'
    ],
}

# -----------------------------
# B. LOAD AND VALIDATE INPUTS
# -----------------------------
def load_and_validate_workbook(path):
    xls = pd.ExcelFile(path)
    checks = []
    missing_sheets = [s for s in REQUIRED_COLUMNS if s not in xls.sheet_names]
    if missing_sheets:
        raise ValueError(f'Missing required sheets: {missing_sheets}')

    data = {s: pd.read_excel(path, sheet_name=s) for s in REQUIRED_COLUMNS}

    for sheet, df in data.items():
        missing_cols = [c for c in REQUIRED_COLUMNS[sheet] if c not in df.columns]
        checks.append({
            'Check': f'{sheet}: required columns',
            'Status': 'PASS' if not missing_cols else 'FAIL',
            'Details': 'All required columns found' if not missing_cols else f'Missing: {missing_cols}'
        })
        if missing_cols:
            raise ValueError(f'{sheet} is missing columns: {missing_cols}')

        df['Month'] = pd.to_datetime(df['Month'], errors='coerce').dt.to_period('M').dt.to_timestamp()
        invalid_months = int(df['Month'].isna().sum())
        checks.append({
            'Check': f'{sheet}: valid months',
            'Status': 'PASS' if invalid_months == 0 else 'FAIL',
            'Details': f'{invalid_months} invalid month value(s)'
        })

        duplicate_months = int(df['Month'].duplicated().sum())
        checks.append({
            'Check': f'{sheet}: duplicate months',
            'Status': 'PASS' if duplicate_months == 0 else 'FAIL',
            'Details': f'{duplicate_months} duplicate month(s)'
        })

        missing_values = int(df[REQUIRED_COLUMNS[sheet]].isna().sum().sum())
        checks.append({
            'Check': f'{sheet}: missing values',
            'Status': 'PASS' if missing_values == 0 else 'FAIL',
            'Details': f'{missing_values} missing required value(s)'
        })

        numeric_cols = [c for c in REQUIRED_COLUMNS[sheet] if c != 'Month']
        for c in numeric_cols:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        negative_count = int((df[numeric_cols] < 0).sum().sum())
        checks.append({
            'Check': f'{sheet}: negative values',
            'Status': 'PASS' if negative_count == 0 else 'FAIL',
            'Details': f'{negative_count} negative volume/cost/assumption value(s)'
        })

        if invalid_months or duplicate_months or missing_values or negative_count:
            raise ValueError(f'Critical validation error in {sheet}. Review the validation messages above.')

        data[sheet] = df.sort_values('Month').reset_index(drop=True)

    fc = data['Forecast_Assumptions']
    checks.append({
        'Check': 'Forecast horizon',
        'Status': 'PASS' if len(fc) == 12 else 'WARN',
        'Details': f'{len(fc)} forecast month(s); model requested 12'
    })
    return data, pd.DataFrame(checks)


def safe_divide(numerator, denominator, default=np.nan):
    denominator = np.asarray(denominator)
    return np.where(denominator != 0, np.asarray(numerator) / denominator, default)


def fit_cost_driver(y, x):
    """Simple y = intercept + slope*x regression, constrained to non-negative components."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or np.nanstd(x) == 0:
        return float(np.nanmean(y)), 0.0, 0.0
    slope, intercept = np.polyfit(x, y, 1)
    slope = max(0.0, float(slope))
    intercept = max(0.0, float(intercept))
    pred = intercept + slope * x
    ss_res = np.nansum((y - pred) ** 2)
    ss_tot = np.nansum((y - np.nanmean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return intercept, slope, float(r2)


data, validation_checks = load_and_validate_workbook(INPUT_FILE)
ops = data['Historical_Operations'].copy()
costs = data['Historical_Costs'].copy()
fcst = data['Forecast_Assumptions'].copy()

hist = ops.merge(costs, on='Month', how='inner', validate='one_to_one')
if len(hist) != len(ops) or len(hist) != len(costs):
    raise ValueError('Historical Operations and Historical Costs months do not fully reconcile.')

# -----------------------------
# C. HISTORICAL DRIVER CALIBRATION
# -----------------------------
hist['Total_Processed_Cases'] = hist['Cases_Received'] + hist['Cases_Shipped']
hist['Cases_Shipped_Per_Labour_Hour'] = safe_divide(hist['Cases_Shipped'], hist['Labour_Hours'])
hist['Total_Cases_Per_Labour_Hour'] = safe_divide(hist['Total_Processed_Cases'], hist['Labour_Hours'])
hist['Labour_Cost_EUR'] = (
    hist['Permanent_Labour_Cost_EUR'] + hist['Temporary_Labour_Cost_EUR'] + hist['Overtime_Cost_EUR']
)
hist['Labour_Cost_Per_Hour_EUR'] = safe_divide(hist['Labour_Cost_EUR'], hist['Labour_Hours'])
hist['Permanent_Cost_Per_FTE_EUR'] = safe_divide(hist['Permanent_Labour_Cost_EUR'], hist['FTE_Count'])
hist['Overtime_Cost_Per_Hour_EUR'] = safe_divide(hist['Overtime_Cost_EUR'], hist['Overtime_Hours'])
hist['Transport_Cost_Per_Load_EUR'] = safe_divide(hist['Transport_Cost_EUR'], hist['Transport_Loads'])
hist['Cases_Per_Load'] = safe_divide(hist['Cases_Shipped'], hist['Transport_Loads'])
hist['Energy_Cost_Per_Pallet_EUR'] = safe_divide(hist['Energy_Cost_EUR'], hist['Pallets_Stored'])
hist['Maintenance_Cost_Per_Case_EUR'] = safe_divide(hist['Maintenance_Cost_EUR'], hist['Cases_Shipped'])

lookback = min(MODEL_SETTINGS['historical_driver_lookback_months'], len(hist))
h = hist.tail(lookback).copy()
base_fte = float(hist.tail(MODEL_SETTINGS['baseline_fte_lookback_months'])['FTE_Count'].mean())
base_perm_cost_per_fte = float(h['Permanent_Cost_Per_FTE_EUR'].median())
base_ot_cost_per_hour = float(h['Overtime_Cost_Per_Hour_EUR'].replace([np.inf, -np.inf], np.nan).median())
base_cases_per_load = float(h['Cases_Per_Load'].median())
base_transport_cost_per_load = float(h['Transport_Cost_Per_Load_EUR'].median())
base_other_cost = float(h['Other_Cost_EUR'].median())
base_rent_cost = float(hist.iloc[-1]['Rent_Cost_EUR'])
base_perm_hourly_cost = base_perm_cost_per_fte / MODEL_SETTINGS['standard_hours_per_fte_month']
base_temp_cost_per_hour = base_perm_hourly_cost * (1 + MODEL_SETTINGS['temporary_labour_premium_vs_permanent'])

# Estimate historical storage turns using the provided forecast cases-per-pallet assumption.
reference_cases_per_pallet = float(fcst['Average_Cases_Per_Pallet'].median())
h['Pallet_Turns_Per_Month'] = safe_divide(
    (h['Cases_Received'] + h['Cases_Shipped']) / 2,
    h['Pallets_Stored'] * reference_cases_per_pallet
)
base_pallet_turns = float(np.nanmedian(h['Pallet_Turns_Per_Month']))

energy_fixed, energy_per_pallet, energy_r2 = fit_cost_driver(h['Energy_Cost_EUR'], h['Pallets_Stored'])
maint_fixed, maint_per_case, maint_r2 = fit_cost_driver(h['Maintenance_Cost_EUR'], h['Cases_Shipped'])

historical_kpis = pd.DataFrame([
    ['Labour cost per hour', float(h['Labour_Cost_Per_Hour_EUR'].median()), 'EUR/hour'],
    ['Cases shipped per labour hour', float(h['Cases_Shipped_Per_Labour_Hour'].median()), 'cases/hour'],
    ['Total processed cases per labour hour', float(h['Total_Cases_Per_Labour_Hour'].median()), 'cases/hour'],
    ['Transport cost per load', base_transport_cost_per_load, 'EUR/load'],
    ['Cases per transport load', base_cases_per_load, 'cases/load'],
    ['Energy cost per pallet', float(h['Energy_Cost_Per_Pallet_EUR'].median()), 'EUR/pallet'],
    ['Maintenance cost per shipped case', float(h['Maintenance_Cost_Per_Case_EUR'].median()), 'EUR/case'],
    ['Pallet turns per month', base_pallet_turns, 'turns/month'],
    ['Baseline permanent FTE', base_fte, 'FTE'],
], columns=['Historical_Driver', 'Value', 'Unit'])

# -----------------------------
# D. SCENARIO FORECAST ENGINE
# -----------------------------
def build_scenario(name, volume_change):
    df = fcst.copy()
    df['Scenario'] = name
    df['Volume_Change_Percent'] = volume_change
    df['Cases_Shipped'] = df['Forecast_Cases_Shipped'] * (1 + volume_change)
    df['Cases_Received'] = df['Forecast_Cases_Received'] * (1 + volume_change)
    df['Total_Processed_Cases'] = df['Cases_Shipped'] + df['Cases_Received']
    df['Required_Labour_Hours'] = df['Total_Processed_Cases'] / df['Productivity_Cases_Per_Hour']
    df['Required_FTEs'] = df['Required_Labour_Hours'] / MODEL_SETTINGS['standard_hours_per_fte_month']

    regular_capacity_hours = base_fte * MODEL_SETTINGS['standard_hours_per_fte_month']
    max_ot_hours = base_fte * MODEL_SETTINGS['max_overtime_hours_per_fte_month']
    df['Regular_Hours_Available'] = regular_capacity_hours
    df['Overtime_Hours'] = np.minimum(np.maximum(df['Required_Labour_Hours'] - regular_capacity_hours, 0), max_ot_hours)
    df['Temporary_Labour_Hours'] = np.maximum(
        df['Required_Labour_Hours'] - regular_capacity_hours - df['Overtime_Hours'], 0
    )
    df['Temporary_FTEs'] = df['Temporary_Labour_Hours'] / MODEL_SETTINGS['standard_hours_per_fte_month']

    # Storage proxy: average monthly flow divided by cases per pallet and historical pallet turns.
    avg_flow = (df['Cases_Received'] + df['Cases_Shipped']) / 2
    df['Pallets_Stored'] = avg_flow / (df['Average_Cases_Per_Pallet'] * base_pallet_turns)
    df['Capacity_Utilisation_Percent'] = df['Pallets_Stored'] / df['Maximum_Pallet_Capacity']
    df['Transport_Loads'] = df['Cases_Shipped'] / base_cases_per_load

    months_from_base = np.arange(1, len(df) + 1)
    salary_factor = (1 + df['Salary_Inflation_Percent']) ** (months_from_base / 12)
    energy_factor = (1 + df['Energy_Inflation_Percent']) ** (months_from_base / 12)
    transport_factor = (1 + df['Transport_Inflation_Percent']) ** (months_from_base / 12)

    df['Permanent_Labour_Cost_EUR'] = base_fte * base_perm_cost_per_fte * salary_factor
    df['Temporary_Labour_Cost_EUR'] = df['Temporary_Labour_Hours'] * base_temp_cost_per_hour * salary_factor
    df['Overtime_Cost_EUR'] = df['Overtime_Hours'] * base_ot_cost_per_hour * salary_factor
    df['Energy_Fixed_Cost_EUR'] = energy_fixed * energy_factor
    df['Energy_Variable_Cost_EUR'] = energy_per_pallet * df['Pallets_Stored'] * energy_factor
    df['Energy_Cost_EUR'] = df['Energy_Fixed_Cost_EUR'] + df['Energy_Variable_Cost_EUR']
    df['Maintenance_Fixed_Cost_EUR'] = maint_fixed
    df['Maintenance_Variable_Cost_EUR'] = maint_per_case * df['Cases_Shipped']
    df['Maintenance_Cost_EUR'] = df['Maintenance_Fixed_Cost_EUR'] + df['Maintenance_Variable_Cost_EUR']
    df['Rent_Cost_EUR'] = base_rent_cost
    df['Transport_Cost_EUR'] = df['Transport_Loads'] * base_transport_cost_per_load * transport_factor
    df['Other_Cost_EUR'] = base_other_cost

    cost_cols = [
        'Permanent_Labour_Cost_EUR', 'Temporary_Labour_Cost_EUR', 'Overtime_Cost_EUR',
        'Energy_Cost_EUR', 'Maintenance_Cost_EUR', 'Rent_Cost_EUR',
        'Transport_Cost_EUR', 'Other_Cost_EUR'
    ]
    df['Total_DC_Cost_EUR'] = df[cost_cols].sum(axis=1)
    df['Cost_Per_Case_Shipped_EUR'] = df['Total_DC_Cost_EUR'] / df['Cases_Shipped']
    df['Fixed_Cost_EUR'] = (
        df['Permanent_Labour_Cost_EUR'] + df['Energy_Fixed_Cost_EUR'] +
        df['Maintenance_Fixed_Cost_EUR'] + df['Rent_Cost_EUR'] + df['Other_Cost_EUR']
    )
    df['Variable_Cost_EUR'] = (
        df['Temporary_Labour_Cost_EUR'] + df['Overtime_Cost_EUR'] +
        df['Energy_Variable_Cost_EUR'] + df['Maintenance_Variable_Cost_EUR'] +
        df['Transport_Cost_EUR']
    )
    df['Cost_Reconciliation_Difference_EUR'] = df['Total_DC_Cost_EUR'] - (df['Fixed_Cost_EUR'] + df['Variable_Cost_EUR'])
    return df

scenario_results = pd.concat([
    build_scenario(s, change) for s, change in MODEL_SETTINGS['scenario_volume_change'].items()
], ignore_index=True)

base_monthly = scenario_results[scenario_results['Scenario'] == 'Base'].copy()

# -----------------------------
# E. VALIDATION, RISKS AND SUMMARY
# -----------------------------
historical_total_productivity = float(h['Total_Cases_Per_Labour_Hour'].median())
prod_low = historical_total_productivity * MODEL_SETTINGS['productivity_warning_low_factor']
prod_high = historical_total_productivity * MODEL_SETTINGS['productivity_warning_high_factor']
unrealistic_productivity = fcst.loc[
    (fcst['Productivity_Cases_Per_Hour'] < prod_low) |
    (fcst['Productivity_Cases_Per_Hour'] > prod_high), ['Month', 'Productivity_Cases_Per_Hour']
]

validation_checks = pd.concat([validation_checks, pd.DataFrame([
    {
        'Check': 'Forecast productivity reasonableness',
        'Status': 'PASS' if unrealistic_productivity.empty else 'WARN',
        'Details': (
            f'All months within {prod_low:.1f}–{prod_high:.1f} cases/hour'
            if unrealistic_productivity.empty else
            f'{len(unrealistic_productivity)} month(s) outside {prod_low:.1f}–{prod_high:.1f}; historical median is {historical_total_productivity:.1f}'
        )
    },
    {
        'Check': 'Capacity utilisation above 100%',
        'Status': 'PASS' if not (scenario_results['Capacity_Utilisation_Percent'] > 1).any() else 'WARN',
        'Details': f"{int((scenario_results['Capacity_Utilisation_Percent'] > 1).sum())} scenario-month(s) above capacity"
    },
    {
        'Check': 'Forecast cost reconciliation',
        'Status': 'PASS' if scenario_results['Cost_Reconciliation_Difference_EUR'].abs().max() < 0.01 else 'FAIL',
        'Details': f"Maximum difference: EUR {scenario_results['Cost_Reconciliation_Difference_EUR'].abs().max():,.4f}"
    },
    {
        'Check': 'Forecast total consistency',
        'Status': 'PASS' if np.isfinite(scenario_results['Total_DC_Cost_EUR']).all() else 'FAIL',
        'Details': 'All scenario totals are finite and calculated'
    },
])], ignore_index=True)

scenario_summary = scenario_results.groupby('Scenario', sort=False).agg(
    Total_Cases_Shipped=('Cases_Shipped', 'sum'),
    Total_DC_Cost_EUR=('Total_DC_Cost_EUR', 'sum'),
    Average_Cost_Per_Case_EUR=('Cost_Per_Case_Shipped_EUR', 'mean'),
    Peak_Required_FTEs=('Required_FTEs', 'max'),
    Peak_Overtime_Hours=('Overtime_Hours', 'max'),
    Peak_Temporary_FTEs=('Temporary_FTEs', 'max'),
    Peak_Capacity_Utilisation_Percent=('Capacity_Utilisation_Percent', 'max')
).reset_index()

risks = []
peak_cap = scenario_results.loc[scenario_results['Capacity_Utilisation_Percent'].idxmax()]
risks.append({
    'Risk_Area': 'Storage capacity',
    'Severity': 'High' if peak_cap['Capacity_Utilisation_Percent'] > 1 else ('Medium' if peak_cap['Capacity_Utilisation_Percent'] > 0.85 else 'Low'),
    'Finding': f"Peak utilisation is {peak_cap['Capacity_Utilisation_Percent']:.1%} in {peak_cap['Month']:%b %Y} ({peak_cap['Scenario']}).",
    'Management_Action': 'Validate inventory policy, pallet turns and overflow capacity before committing to the volume plan.'
})
peak_fte = scenario_results.loc[scenario_results['Required_FTEs'].idxmax()]
risks.append({
    'Risk_Area': 'Labour capacity',
    'Severity': 'High' if peak_fte['Temporary_FTEs'] > 0 else ('Medium' if peak_fte['Overtime_Hours'] > 0 else 'Low'),
    'Finding': f"Peak requirement is {peak_fte['Required_FTEs']:.1f} FTE in {peak_fte['Month']:%b %Y} ({peak_fte['Scenario']}); baseline permanent FTE is {base_fte:.1f}.",
    'Management_Action': 'Pre-book agency labour and review shift coverage where overtime or temporary labour is triggered.'
})
risks.append({
    'Risk_Area': 'Productivity assumption',
    'Severity': 'Medium' if not unrealistic_productivity.empty else 'Low',
    'Finding': f"Forecast productivity ranges from {fcst['Productivity_Cases_Per_Hour'].min():.1f} to {fcst['Productivity_Cases_Per_Hour'].max():.1f} versus a historical median of {historical_total_productivity:.1f} total processed cases/hour.",
    'Management_Action': 'Confirm whether the productivity assumption includes both inbound and outbound handling and monitor weekly actuals.'
})
risks.append({
    'Risk_Area': 'Inventory modelling',
    'Severity': 'Medium',
    'Finding': 'Opening inventory and target inventory days are not provided; storage is estimated using historical pallet turns.',
    'Management_Action': 'Add opening pallets, safety stock and monthly closing inventory assumptions for a stronger capacity forecast.'
})
risks.append({
    'Risk_Area': 'Cost-driver reliability',
    'Severity': 'Medium' if min(energy_r2, maint_r2) < 0.30 else 'Low',
    'Finding': f"Energy regression R² is {energy_r2:.2f}; maintenance regression R² is {maint_r2:.2f}.",
    'Management_Action': 'Replace statistical drivers with contractual tariffs or engineering standards when available.'
})
risks_df = pd.DataFrame(risks)

# Model assumptions table for transparency
assumptions_output = pd.DataFrame([
    ['Standard hours per FTE per month', MODEL_SETTINGS['standard_hours_per_fte_month'], 'Editable default; not included in source workbook'],
    ['Maximum overtime hours per FTE per month', MODEL_SETTINGS['max_overtime_hours_per_fte_month'], 'Editable default; not included in source workbook'],
    ['Temporary labour premium vs permanent', MODEL_SETTINGS['temporary_labour_premium_vs_permanent'], 'Applied because temporary hours/rate are not supplied'],
    ['Baseline permanent FTE', base_fte, 'Average of latest historical months'],
    ['Permanent cost per FTE per month', base_perm_cost_per_fte, 'Historical median'],
    ['Overtime cost per hour', base_ot_cost_per_hour, 'Historical median'],
    ['Temporary cost per hour', base_temp_cost_per_hour, 'Permanent hourly cost plus agency premium'],
    ['Cases per transport load', base_cases_per_load, 'Historical median'],
    ['Transport cost per load', base_transport_cost_per_load, 'Historical median'],
    ['Historical pallet turns per month', base_pallet_turns, 'Estimated using forecast cases per pallet'],
    ['Energy fixed cost', energy_fixed, f'Historical regression intercept; R²={energy_r2:.2f}'],
    ['Energy variable cost per pallet', energy_per_pallet, f'Historical regression slope; R²={energy_r2:.2f}'],
    ['Maintenance fixed cost', maint_fixed, f'Historical regression intercept; R²={maint_r2:.2f}'],
    ['Maintenance variable cost per shipped case', maint_per_case, f'Historical regression slope; R²={maint_r2:.2f}'],
    ['Rent cost per month', base_rent_cost, 'Latest historical month'],
    ['Other cost per month', base_other_cost, 'Historical median'],
], columns=['Assumption', 'Value', 'Basis'])

# -----------------------------
# F. EXCEL OUTPUT
# -----------------------------
def write_excel_output(path):
    with pd.ExcelWriter(path, engine='xlsxwriter', datetime_format='mmm-yy') as writer:
        base_monthly.to_excel(writer, sheet_name='Monthly_Base_Forecast', index=False)
        scenario_summary.to_excel(writer, sheet_name='Scenario_Comparison', index=False)
        scenario_results.to_excel(writer, sheet_name='All_Scenarios', index=False)
        historical_kpis.to_excel(writer, sheet_name='Historical_Drivers', index=False)
        assumptions_output.to_excel(writer, sheet_name='Model_Assumptions', index=False)
        risks_df.to_excel(writer, sheet_name='Risks_Constraints', index=False)
        validation_checks.to_excel(writer, sheet_name='Validation_Checks', index=False)
        hist.to_excel(writer, sheet_name='Historical_Calculations', index=False)
        fcst.to_excel(writer, sheet_name='Input_Forecast_Assumptions', index=False)

        workbook = writer.book
        header_fmt = workbook.add_format({'bold': True, 'font_color': 'white', 'bg_color': '#102A43', 'border': 0})
        input_fmt = workbook.add_format({'font_color': '#0000FF', 'num_format': '#,##0.00;[Red](#,##0.00);-'})
        calc_fmt = workbook.add_format({'font_color': '#000000', 'num_format': '#,##0.00;[Red](#,##0.00);-'})
        pct_fmt = workbook.add_format({'num_format': '0.0%', 'font_color': '#000000'})
        euro_fmt = workbook.add_format({'num_format': '€#,##0;[Red](€#,##0);-', 'font_color': '#000000'})
        euro2_fmt = workbook.add_format({'num_format': '€0.00;[Red](€0.00);-', 'font_color': '#000000'})
        date_fmt = workbook.add_format({'num_format': 'mmm-yy'})
        warn_fmt = workbook.add_format({'bg_color': '#FFF2CC', 'font_color': '#9C6500'})
        fail_fmt = workbook.add_format({'bg_color': '#F4CCCC', 'font_color': '#9C0006'})
        pass_fmt = workbook.add_format({'bg_color': '#D9EAD3', 'font_color': '#274E13'})

        for sheet_name, df in {
            'Monthly_Base_Forecast': base_monthly,
            'Scenario_Comparison': scenario_summary,
            'All_Scenarios': scenario_results,
            'Historical_Drivers': historical_kpis,
            'Model_Assumptions': assumptions_output,
            'Risks_Constraints': risks_df,
            'Validation_Checks': validation_checks,
            'Historical_Calculations': hist,
            'Input_Forecast_Assumptions': fcst,
        }.items():
            ws = writer.sheets[sheet_name]
            ws.freeze_panes(1, 1)
            ws.hide_gridlines(2)
            ws.set_row(0, 24, header_fmt)
            ws.autofilter(0, 0, len(df), max(0, len(df.columns)-1))
            for col_idx, col in enumerate(df.columns):
                width = min(max(len(str(col)) + 2, 12), 34)
                if df[col].dtype == 'object':
                    sample_len = df[col].astype(str).str.len().quantile(0.90) if len(df) else 10
                    width = min(max(width, int(sample_len) + 2), 55)
                ws.set_column(col_idx, col_idx, width)
                if 'Month' == col or col.endswith('_Month'):
                    ws.set_column(col_idx, col_idx, 12, date_fmt)
                elif 'Percent' in col or 'Utilisation' in col or 'Inflation' in col or 'Change' in col:
                    ws.set_column(col_idx, col_idx, width, pct_fmt)
                elif 'EUR' in col or 'Cost' in col:
                    ws.set_column(col_idx, col_idx, width, euro2_fmt if 'Per_' in col or 'per' in col.lower() else euro_fmt)
                elif pd.api.types.is_numeric_dtype(df[col]):
                    ws.set_column(col_idx, col_idx, width, calc_fmt)
            if sheet_name == 'Input_Forecast_Assumptions':
                ws.set_column(1, len(df.columns)-1, 24, input_fmt)
            if sheet_name == 'Validation_Checks':
                status_col = df.columns.get_loc('Status')
                ws.conditional_format(1, status_col, len(df), status_col, {'type': 'text', 'criteria': 'containing', 'value': 'PASS', 'format': pass_fmt})
                ws.conditional_format(1, status_col, len(df), status_col, {'type': 'text', 'criteria': 'containing', 'value': 'WARN', 'format': warn_fmt})
                ws.conditional_format(1, status_col, len(df), status_col, {'type': 'text', 'criteria': 'containing', 'value': 'FAIL', 'format': fail_fmt})

        # Capacity conditional formatting in key output sheets.
        for sheet_name, df in [('Monthly_Base_Forecast', base_monthly), ('All_Scenarios', scenario_results)]:
            ws = writer.sheets[sheet_name]
            c = df.columns.get_loc('Capacity_Utilisation_Percent')
            ws.conditional_format(1, c, len(df), c, {'type': 'cell', 'criteria': '>', 'value': 1, 'format': fail_fmt})
            ws.conditional_format(1, c, len(df), c, {'type': 'between', 'minimum': 0.85, 'maximum': 1, 'format': warn_fmt})

write_excel_output(OUTPUT_EXCEL)

# -----------------------------
# G. INTERACTIVE HTML REPORT
# -----------------------------
def money(x): return f'€{x:,.0f}'
def number(x, d=1): return f'{x:,.{d}f}'
def pct(x): return f'{x:.1%}'

base_summary = scenario_summary[scenario_summary['Scenario'] == 'Base'].iloc[0]
upside_summary = scenario_summary[scenario_summary['Scenario'] == 'Upside'].iloc[0]

fig_cost = go.Figure()
for scenario in ['Base', 'Upside', 'Downside']:
    d = scenario_results[scenario_results['Scenario'] == scenario]
    fig_cost.add_trace(go.Scatter(x=d['Month'], y=d['Total_DC_Cost_EUR'], mode='lines+markers', name=scenario))
fig_cost.update_layout(title='Monthly Total Distribution Center Cost', yaxis_title='EUR', template='plotly_white', height=390, margin=dict(l=40,r=20,t=60,b=40))

fig_cpc = go.Figure()
for scenario in ['Base', 'Upside', 'Downside']:
    d = scenario_results[scenario_results['Scenario'] == scenario]
    fig_cpc.add_trace(go.Scatter(x=d['Month'], y=d['Cost_Per_Case_Shipped_EUR'], mode='lines+markers', name=scenario))
fig_cpc.update_layout(title='Cost per Case Shipped', yaxis_title='EUR per case', template='plotly_white', height=390, margin=dict(l=40,r=20,t=60,b=40))

fig_fte = go.Figure()
for scenario in ['Base', 'Upside', 'Downside']:
    d = scenario_results[scenario_results['Scenario'] == scenario]
    fig_fte.add_trace(go.Scatter(x=d['Month'], y=d['Required_FTEs'], mode='lines+markers', name=scenario))
fig_fte.add_hline(y=base_fte, line_dash='dash', annotation_text='Baseline permanent FTE')
fig_fte.update_layout(title='Required Warehouse FTEs', yaxis_title='FTE', template='plotly_white', height=390, margin=dict(l=40,r=20,t=60,b=40))

fig_cap = go.Figure()
for scenario in ['Base', 'Upside', 'Downside']:
    d = scenario_results[scenario_results['Scenario'] == scenario]
    fig_cap.add_trace(go.Scatter(x=d['Month'], y=d['Capacity_Utilisation_Percent'], mode='lines+markers', name=scenario))
fig_cap.add_hline(y=0.85, line_dash='dot', annotation_text='85% watch level')
fig_cap.add_hline(y=1.00, line_dash='dash', annotation_text='100% capacity')
fig_cap.update_layout(title='Storage Capacity Utilisation', yaxis_tickformat='.0%', template='plotly_white', height=390, margin=dict(l=40,r=20,t=60,b=40))

scenario_html = scenario_summary.copy()
scenario_html['Total_Cases_Shipped'] = scenario_html['Total_Cases_Shipped'].map(lambda x: f'{x:,.0f}')
scenario_html['Total_DC_Cost_EUR'] = scenario_html['Total_DC_Cost_EUR'].map(money)
scenario_html['Average_Cost_Per_Case_EUR'] = scenario_html['Average_Cost_Per_Case_EUR'].map(lambda x: f'€{x:,.2f}')
scenario_html['Peak_Required_FTEs'] = scenario_html['Peak_Required_FTEs'].map(lambda x: f'{x:,.1f}')
scenario_html['Peak_Overtime_Hours'] = scenario_html['Peak_Overtime_Hours'].map(lambda x: f'{x:,.0f}')
scenario_html['Peak_Temporary_FTEs'] = scenario_html['Peak_Temporary_FTEs'].map(lambda x: f'{x:,.1f}')
scenario_html['Peak_Capacity_Utilisation_Percent'] = scenario_html['Peak_Capacity_Utilisation_Percent'].map(pct)

monthly_display_cols = [
    'Month','Cases_Shipped','Cases_Received','Required_Labour_Hours','Required_FTEs',
    'Overtime_Hours','Temporary_FTEs','Pallets_Stored','Capacity_Utilisation_Percent',
    'Total_DC_Cost_EUR','Cost_Per_Case_Shipped_EUR'
]
monthly_html = base_monthly[monthly_display_cols].copy()
monthly_html['Month'] = monthly_html['Month'].dt.strftime('%b-%Y')
for c in ['Cases_Shipped','Cases_Received','Required_Labour_Hours','Overtime_Hours','Pallets_Stored']:
    monthly_html[c] = monthly_html[c].map(lambda x: f'{x:,.0f}')
for c in ['Required_FTEs','Temporary_FTEs']:
    monthly_html[c] = monthly_html[c].map(lambda x: f'{x:,.1f}')
monthly_html['Capacity_Utilisation_Percent'] = monthly_html['Capacity_Utilisation_Percent'].map(pct)
monthly_html['Total_DC_Cost_EUR'] = monthly_html['Total_DC_Cost_EUR'].map(money)
monthly_html['Cost_Per_Case_Shipped_EUR'] = monthly_html['Cost_Per_Case_Shipped_EUR'].map(lambda x: f'€{x:,.2f}')

risk_cards = ''.join([
    f"<div class='risk {r.Severity.lower()}'><div class='risk-head'><b>{r.Risk_Area}</b><span>{r.Severity}</span></div><p>{r.Finding}</p><small>{r.Management_Action}</small></div>"
    for r in risks_df.itertuples()
])

html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Distribution Center CFO Forecast</title>
<style>
:root{{--navy:#102A43;--blue:#1769AA;--red:#D64545;--light:#F4F7FA;--ink:#243B53;--green:#2E7D32;}}
*{{box-sizing:border-box}} body{{margin:0;background:var(--light);font-family:Inter,Arial,sans-serif;color:var(--ink)}}
.hero{{background:linear-gradient(120deg,#081F33,#1769AA);color:white;padding:44px 6vw 38px}} .hero h1{{font-size:38px;margin:0 0 10px}} .hero p{{max-width:920px;opacity:.88;margin:0}}
.wrap{{max-width:1450px;margin:auto;padding:28px 4vw 60px}} .kpis{{display:grid;grid-template-columns:repeat(5,minmax(160px,1fr));gap:16px;margin-top:-52px}}
.card{{background:white;border-radius:14px;padding:20px;box-shadow:0 10px 28px rgba(16,42,67,.10);border-top:4px solid var(--blue)}} .card .label{{font-size:12px;text-transform:uppercase;letter-spacing:.08em;color:#627D98}} .card .value{{font-size:26px;font-weight:800;margin-top:7px;color:var(--navy)}}
.section{{background:white;border-radius:14px;padding:24px;margin-top:22px;box-shadow:0 8px 24px rgba(16,42,67,.07)}} h2{{margin:0 0 18px;color:var(--navy)}}
.grid2{{display:grid;grid-template-columns:1fr 1fr;gap:20px}} table{{border-collapse:collapse;width:100%;font-size:13px}} th{{background:var(--navy);color:white;text-align:left;padding:10px;position:sticky;top:0}} td{{padding:9px 10px;border-bottom:1px solid #D9E2EC}} tr:nth-child(even){{background:#F7FAFC}} .table-scroll{{overflow:auto;max-height:540px}}
.risks{{display:grid;grid-template-columns:repeat(2,1fr);gap:14px}} .risk{{border-left:5px solid #829AB1;background:#F8FAFC;padding:16px;border-radius:10px}} .risk.high{{border-color:#D64545}} .risk.medium{{border-color:#F0A202}} .risk.low{{border-color:#2E7D32}} .risk-head{{display:flex;justify-content:space-between}} .risk-head span{{font-size:12px;font-weight:700;text-transform:uppercase}} .risk p{{margin:8px 0}} .risk small{{color:#627D98}}
.note{{padding:14px 16px;background:#FFF8E1;border-left:5px solid #F0A202;border-radius:8px;margin-top:16px}} footer{{text-align:center;color:#829AB1;padding:30px}}
@media(max-width:1000px){{.kpis{{grid-template-columns:repeat(2,1fr);margin-top:20px}}.grid2,.risks{{grid-template-columns:1fr}}}} @media(max-width:600px){{.kpis{{grid-template-columns:1fr}}.hero h1{{font-size:29px}}}}
</style></head><body>
<div class='hero'><h1>Distribution Center 12-Month CFO Forecast</h1><p>Volume, labour, operating cost and capacity outlook across Base, Upside and Downside scenarios.</p></div>
<div class='wrap'>
<div class='kpis'>
<div class='card'><div class='label'>Base annual cost</div><div class='value'>{money(base_summary.Total_DC_Cost_EUR)}</div></div>
<div class='card'><div class='label'>Average cost / case</div><div class='value'>€{base_summary.Average_Cost_Per_Case_EUR:,.2f}</div></div>
<div class='card'><div class='label'>Peak required FTE</div><div class='value'>{base_summary.Peak_Required_FTEs:,.1f}</div></div>
<div class='card'><div class='label'>Peak capacity</div><div class='value'>{pct(base_summary.Peak_Capacity_Utilisation_Percent)}</div></div>
<div class='card'><div class='label'>Upside cost impact</div><div class='value'>{money(upside_summary.Total_DC_Cost_EUR-base_summary.Total_DC_Cost_EUR)}</div></div>
</div>
<div class='section'><h2>Scenario comparison</h2>{scenario_html.to_html(index=False, classes='dataframe', border=0)}</div>
<div class='grid2'>
<div class='section'>{fig_cost.to_html(full_html=False, include_plotlyjs='cdn')}</div>
<div class='section'>{fig_cpc.to_html(full_html=False, include_plotlyjs=False)}</div>
<div class='section'>{fig_fte.to_html(full_html=False, include_plotlyjs=False)}</div>
<div class='section'>{fig_cap.to_html(full_html=False, include_plotlyjs=False)}</div>
</div>
<div class='section'><h2>Key risks and constraints</h2><div class='risks'>{risk_cards}</div><div class='note'><b>Model limitation:</b> pallet storage uses historical pallet turns because opening inventory and target inventory days were not supplied.</div></div>
<div class='section'><h2>Base monthly forecast</h2><div class='table-scroll'>{monthly_html.to_html(index=False, classes='dataframe', border=0)}</div></div>
<div class='section'><h2>Model calibration</h2><p>Historical median total throughput productivity: <b>{historical_total_productivity:.1f} cases/hour</b>. Baseline permanent workforce: <b>{base_fte:.1f} FTE</b>. Historical pallet turns: <b>{base_pallet_turns:.2f}x per month</b>.</p><p>Energy regression R²: <b>{energy_r2:.2f}</b>. Maintenance regression R²: <b>{maint_r2:.2f}</b>.</p></div>
<footer>Generated by the Distribution Center FP&A Python model</footer>
</div></body></html>"""

Path(OUTPUT_HTML).write_text(html, encoding='utf-8')

print('\nMODEL COMPLETE')
print(f'Excel output: {OUTPUT_EXCEL}')
print(f'HTML report: {OUTPUT_HTML}')
print('\nValidation checks:')
display(validation_checks)
print('\nScenario comparison:')
display(scenario_summary)
display(HTML(f"<a href='{OUTPUT_HTML}' target='_blank' style='font-size:18px;font-weight:bold'>Open interactive CFO HTML report</a>"))

# Download outputs automatically in Google Colab.
try:
    from google.colab import files
    files.download(OUTPUT_EXCEL)
    files.download(OUTPUT_HTML)
except ImportError:
    pass



## How to update the model

- Change scenario percentages, standard monthly hours, overtime limits or temporary labour premium in `MODEL_SETTINGS`.
- Replace the pallet-turn approach after adding opening inventory, closing inventory or target inventory days.
- Replace regression-based energy and maintenance costs with supplier tariffs or engineering standards when available.
- Keep the workbook sheet and column names unchanged unless you also update `REQUIRED_COLUMNS`.
